# Store Sales - Time Series Forecasting

In [1]:
import joblib
import lightgbm as lgb
import math
import numpy as np
import pandas as pd
import re
import sklearn

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
oil = pd.read_csv("oil.csv")
stores = pd.read_csv("stores.csv")
transactions = pd.read_csv("transactions.csv")
holidays = pd.read_csv("holidays_events.csv")

In [3]:
train["test"] = 0
test["test"] = 1
data = pd.concat([train, test], axis=0)

In [4]:
data = data.merge(holidays, on="date", how="left")
data = data.merge(stores, on="store_nbr", how="left")
data = data.merge(oil, on="date", how="left")
data = data.merge(transactions, on=["date", "store_nbr"], how="left")
data = data.set_index(["store_nbr", "date", "family"])
data = data.drop(index="2013-01-01", level=1, errors="ignore")

In [5]:
data_ = data.copy().reset_index()

In [6]:
grouping_columns = [
    "year",
    "quarter",
    "month",
    "day_of_week",
    "day_of_year",
    "day_of_month",
]

In [7]:
grouped_data = data_.groupby(["store_nbr", "family"])

In [8]:
alphas = [0.95, 0.8, 0.65, 0.5]
lags = [1, 7, 30]

In [9]:
for a in alphas:
  for i in lags:
    data_[f"sales_lag_{i}_alpha_{a}"] = np.log1p(
        grouped_data["sales"].transform(
            lambda x: x.shift(i).ewm(alpha=a, min_periods=1).mean()
        )
    )

In [10]:
data_["date"] = pd.to_datetime(data_["date"])
data_["day_of_week"] = data_["date"].dt.day_of_week
data_["day_of_year"] = data_["date"].dt.dayofyear
data_["day_of_month"] = data_["date"].dt.day
data_["month"] = data_["date"].dt.month
data_["quarter"] = data_["date"].dt.quarter
data_["year"] = data_["date"].dt.year

In [11]:
data_["onpromotion"] = data_["onpromotion"].apply(lambda x: x > 0)
sales_lag_columns = list(data_.filter(like="lag").columns)

In [12]:
to_dummies = [
    "day_of_week",
    "day_of_month",
    "month",
    "quarter",
    "year",
    "store_nbr",
    "type_y",
    "cluster",
    "family",
    "onpromotion",
    "type_x",
    "locale",
    "locale_name",
    "city",
    "state",
]

In [13]:
X = data_.loc[
    :,
    [
        "day_of_week",
        "day_of_month",
        "month",
        "quarter",
        "year",
        "store_nbr",
        "type_y",
        "cluster",
        "family",
        "onpromotion",
        "type_x",
        "locale",
        "locale_name",
        "city",
        "state",
        "test",
        "sales",
        "id",
    ]
    + sales_lag_columns,
]

In [14]:
X[to_dummies] = X[to_dummies].astype("category")

In [15]:
data_train = X[X["test"] == 0]
data_test = X[X["test"] == 1]

In [16]:
training_percentage = 0.8
n = len(data_train)
training_start = 0
training_end = math.floor(n * training_percentage)
validation_start = training_end
validation_end = n

In [17]:
X_train = data_train.loc[training_start:training_end, :].drop(
    ["test", "sales", "id"], axis=1
)

In [18]:
y_train = data_train.loc[training_start:training_end, "sales"]
X_val = data_train.loc[validation_start:validation_end, :].drop(
    ["test", "sales", "id"], axis=1
)
y_val = data_train.loc[validation_start:validation_end, "sales"]

In [19]:
X_train = X_train.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x))
X_val = X_val.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x))

In [20]:
hyper_params = {
    "task": "train",
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": ["l1", "l2"],
    "learning_rate": 0.1,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.7,
    "bagging_freq": 10,
    "verbose": 0,
    "max_depth": 50,
    "num_leaves": 128,
    "max_bin": 512,
}

In [21]:
gbm = lgb.LGBMRegressor(**hyper_params)
gbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l1")

[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=10, subsample_freq=0 will be ignored. Current value: bagging_freq=10
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=10, subsample_freq=0 will be ignored. Current value: bagging_freq=10
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=10, subsample_fr

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,50
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [22]:
joblib.dump(gbm, "store_sales_gbm_model.pkl")
joblib.dump(X_train.columns.tolist(), "store_sales_columns.pkl")

['store_sales_columns.pkl']